## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 47 samples and 13085 columns.
First few rows of the data:


,survival time,survival,sample_title,tissue,gender,age,prognostic subtype,SAMD11,NOC2L,HES4,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,35.8,1.0,2004-950RNAseq,Brain Metastatic Tissue,Male,67.0,Poor,2.254931,4.540211,0.293703,...,6.281492,2.964554,4.949571,7.379952,4.789904,3.788943,4.996821,4.581002,2.818548,4.917973
1,1.0,0.0,2006-262-RNAseq,Brain Metastatic Tissue,Male,55.0,Good,2.600297,3.406464,1.033125,...,7.439998,4.657450,6.926805,8.227490,6.421354,5.732441,7.375770,6.918782,4.549217,6.839389
2,1.1,1.0,2006-76RNASeq,Brain Metastatic Tissue,Male,61.0,Good,2.121720,3.574719,2.551187,...,6.074428,2.635651,4.962224,7.092147,4.835275,4.285369,5.572828,5.513111,3.375943,5.612652
3,3.2,1.0,2007-109RNASeq,Brain Metastatic Tissue,Female,61.0,Good,1.446262,3.401710,2.985803,...,6.902861,3.201464,5.671691,8.078788,5.544196,4.731804,6.312369,5.672232,3.493351,6.291637
4,2.6,1.0,2007-263-RNAseq,Brain Metastatic Tissue,Female,61.0,Good,3.515690,3.857159,1.242937,...,8.280964,5.136008,7.612852,8.958045,6.251670,6.483668,8.213153,7.970868,5.070564,7.805208


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'sample_title'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
